# 04 - Inactivity Prediction and Retention Strategy

**Phases 6 to 9.** Predict which customers will stop buying, combine that with
how much they are worth, and turn it into a ranked list of who to contact.

This is where the project answers its actual question:
*Which customers should the company prioritise for retention, and why?*

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
import config
from feature_engineering import build_snapshot, build_all_snapshots
from data_processing import load_clean, load_cancellations
import model as M, retention_strategy as RS

## 1. Defining the target without leaking the future

This is not subscription data - nobody "cancels". So inactivity is defined
using time:

```
[ 180-day observation window ]  |  [ 90-day future window ]
   features come from here ONLY    target comes from here ONLY
```

* **target = 1** - the customer placed no order in the 90-day future window
* **target = 0** - the customer came back

Three snapshots are built at different cut-off dates:

| Snapshot | Purpose |
|---|---|
| **train** (cut 2011-06-09) | fit the model |
| **test** (cut 2011-09-09) | evaluate it - its future period is entirely *after* the training one |
| **score** (cut 2011-12-10) | the live run: features from the most recent 180 days, no target yet |

### Why the observation window is a fixed length

If the window grew longer at each later cut-off, features like frequency and
total spend would rise simply because more time had been included - and the
model would not transfer between snapshots. Holding it at 180 days everywhere
keeps the features comparable.

In [2]:
snapshots = build_all_snapshots()

  train  cutoff=2011-06-09  obs from 2010-12-11  customers=2,625  inactive=50.1%


  test   cutoff=2011-09-09  obs from 2011-03-13  customers=2,775  inactive=38.5%


  score  cutoff=2011-12-10  obs from 2011-06-13  customers=3,471


The training snapshot is almost perfectly balanced (50.1% inactive), so no
resampling or class rebalancing is needed. The test snapshot's rate is lower
(38.5%) because its future window covers the Christmas peak, when more customers
return. That shift is real and is discussed in the evaluation.

In [3]:
snapshots['train'][config.FEATURE_COLUMNS].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
recency_days,2625.0,57.24,47.38,0.00,19.00,48.00,90.00,178.00
frequency,2625.0,2.64,3.71,1.00,1.00,2.00,3.00,67.00
monetary,2625.0,1190.21,4056.25,2.90,243.72,479.65,1039.67,116135.92
avg_order_value,2625.0,411.28,1577.68,2.90,172.90,296.31,430.38,77183.60
max_order_value,2625.0,547.60,1776.44,2.90,201.94,332.41,550.52,77183.60
unique_products,2625.0,41.05,52.54,1.00,12.00,26.00,52.00,827.00
total_quantity,2625.0,705.47,2823.28,1.00,117.00,265.00,589.00,81632.00
tenure_days,2625.0,103.44,50.84,0.00,64.00,105.00,146.00,178.00
avg_days_between_orders,2625.0,110.62,71.94,0.00,36.00,137.00,180.00,180.00
purchase_trend_ratio,2625.0,1.50,1.06,0.12,0.50,1.33,2.00,13.00


## 2. The features

All 15 are computed strictly inside the observation window.

| Feature | What it captures |
|---|---|
| `recency_days`, `frequency`, `monetary` | the RFM core |
| `avg_order_value`, `max_order_value`, `avg_basket_size` | how big their orders are |
| `unique_products` | how broadly they shop |
| `tenure_days`, `active_weeks` | how established and how spread out their activity is |
| `avg_days_between_orders` | their personal reorder rhythm |
| `purchase_trend_ratio` | orders in the recent half vs the earlier half - momentum |
| `cancel_orders`, `cancel_rate` | cancellation behaviour |
| `is_uk` | domestic vs export |

`purchase_trend_ratio` above 1 means the customer is speeding up; below 1 means
slowing down. Single-order customers get `avg_days_between_orders` set to the
full window length, which correctly reads as "no repeat rhythm observed".

## 3. Train and compare two models

In [4]:
metrics = M.run()

Train: 2,625 customers, 50.1% inactive (cut-off 2011-06-09)
Test : 2,775 customers, 38.5% inactive (cut-off 2011-09-09)  <- out of time



Logistic Regression
  ROC-AUC  train 0.7675 | train 5-fold CV 0.7624 | TEST out-of-time 0.7302
  @0.50  precision 0.538  recall 0.744  F1 0.625  accuracy 0.656



Random Forest
  ROC-AUC  train 0.8237 | train 5-fold CV 0.7558 | TEST out-of-time 0.7271
  @0.50  precision 0.538  recall 0.755  F1 0.628  accuracy 0.656

Selected model: Logistic Regression (highest out-of-time ROC-AUC)


Precision / recall trade-off (out-of-time test set):
 threshold  customers_flagged  pct_of_base_flagged  precision  recall    f1
      0.30               2070                 74.6      0.476   0.923 0.628
      0.35               1933                 69.7      0.491   0.889 0.633
      0.40               1778                 64.1      0.506   0.843 0.633
      0.45               1612                 58.1      0.525   0.793 0.632
      0.50               1475                 53.2      0.538   0.744 0.625
      0.55               1342                 48.4      0.550   0.692 0.613
      0.60               1188                 42.8      0.568   0.633 0.599
      0.65                943                 34.0      0.584   0.516 0.548
      0.70                648                 23.4      0.602   0.366 0.455
      0.75                349                 12.6      0.628   0.205 0.309
      0.80                139                  5.0      0.727   0.095 0.167

Campaign capacity - contacting the


Top drivers of inactivity risk:
                feature  importance  coefficient pushes_towards
           active_weeks     0.13900      -1.3848 staying active
        avg_order_value     0.05854      -0.6313 staying active
        max_order_value     0.03441       0.8183     inactivity
avg_days_between_orders     0.01875       0.3355     inactivity
               monetary     0.00796      -0.2293 staying active
        avg_basket_size     0.00775      -0.2359 staying active
        unique_products     0.00626      -0.1408 staying active
              frequency     0.00492       0.2367     inactivity

Scored 3,471 live customers | absolute risk bands: {'Low': 1450, 'Medium': 1439, 'High': 582}


### Reading the results

**Logistic Regression is selected** - it scores slightly higher out-of-time
(ROC-AUC 0.730 vs 0.727) and is far more stable. Look at the gap between the
training and test scores:

| Model | Train | Train CV | Test (out-of-time) |
|---|---|---|---|
| Logistic Regression | 0.768 | 0.762 | **0.730** |
| Random Forest | 0.824 | 0.756 | 0.727 |

The Random Forest looks much stronger on the training data, but its
cross-validated score drops to roughly the same place - it was memorising, not
learning. Choosing it on training accuracy alone would have been a mistake. The
simpler model also comes with readable coefficients, which the business can
interrogate.

An AUC of 0.73 is a genuine but imperfect signal: it ranks customers
substantially better than chance, and that is enough to prioritise a campaign.
It is not a crystal ball, and it should not be sold as one.

### Why accuracy is the wrong headline

At the 0.50 cut-off the model catches **74% of the customers who really do go
inactive** (recall) at 54% precision. Accuracy is only 66%, but accuracy is not
the goal: missing a valuable customer who is about to leave costs their future
revenue, while wrongly flagging someone costs an email.

Recall-weighted tuning (F2) was tried and **rejected** - because the training
classes are balanced, F2 is maximised by flagging almost everybody, which is not
a campaign anyone can run. The trade-off is shown explicitly instead.

In [5]:
pd.read_csv(config.REPORTS_DIR / 'threshold_tradeoff.csv')

,threshold,customers_flagged,pct_of_base_flagged,precision,recall,f1
0,0.30,2070,74.6,0.476,0.923,0.628
1,0.35,1933,69.7,0.491,0.889,0.633
2,0.40,1778,64.1,0.506,0.843,0.633
3,0.45,1612,58.1,0.525,0.793,0.632
4,0.50,1475,53.2,0.538,0.744,0.625
5,0.55,1342,48.4,0.550,0.692,0.613
6,0.60,1188,42.8,0.568,0.633,0.599
7,0.65,943,34.0,0.584,0.516,0.548
8,0.70,648,23.4,0.602,0.366,0.455
9,0.75,349,12.6,0.628,0.205,0.309


### The measure that actually matters to a marketing team

A retention team has limited capacity. The real question is: *if we can contact
N customers, how many of the ones who would have left do we catch?*

In [6]:
pd.read_csv(config.REPORTS_DIR / 'campaign_lift.csv')

,customers_contacted,pct_of_base_contacted,inactive_customers_caught,recall,precision,lift_vs_random
0,250,9.0,161,0.151,0.644,1.67
1,500,18.0,311,0.291,0.622,1.62
2,750,27.0,446,0.418,0.595,1.55
3,1000,36.0,581,0.545,0.581,1.51
4,1500,54.1,804,0.754,0.536,1.39


Contacting the 500 riskiest customers finds 311 who genuinely went inactive -
**1.62x better than contacting 500 customers at random**. That is the concrete
value of the model.

## 4. What drives inactivity?

In [7]:
imp = pd.read_csv(config.REPORTS_DIR / 'feature_importance.csv')
imp.head(10)

,feature,importance,coefficient,pushes_towards
0,active_weeks,0.13900,-1.3848,staying active
1,avg_order_value,0.05854,-0.6313,staying active
2,max_order_value,0.03441,0.8183,inactivity
3,avg_days_between_orders,0.01875,0.3355,inactivity
4,monetary,0.00796,-0.2293,staying active
5,avg_basket_size,0.00775,-0.2359,staying active
6,unique_products,0.00626,-0.1408,staying active
7,frequency,0.00492,0.2367,inactivity
8,recency_days,0.00234,0.0729,inactivity
9,cancel_orders,0.00161,-0.0682,staying active


### Interpreting the drivers

* **`active_weeks` dominates.** Customers whose buying is spread across many
  distinct weeks stay; customers who bought everything in one burst leave.
  Habit matters more than size.
* **`avg_order_value` protects**, but **`max_order_value` raises risk.** These
  pull in opposite directions, and the combination is meaningful: holding
  average order value fixed, a much larger *maximum* order marks a one-off bulk
  purchase - a customer who appeared, bought heavily once, and did not return.
* **`avg_days_between_orders`** - long gaps between past orders predict another
  long gap.
* Cancellations barely matter, which matches the EDA finding that cancelling is a
  sign of an active wholesale relationship rather than dissatisfaction.

## 5. Customer value (Phase 7)

A deliberately simple, transparent score - no probabilistic lifetime-value model,
because a retention manager has to be able to explain it:

```
value score = 0.5 x spend quintile + 0.3 x order-count quintile + 0.2 x basket-size quintile
```

High >= 4.0, Medium 2.5-4.0, Low < 2.5.

## 6. Combining value and risk (Phase 8)

Here the project hit a real problem worth explaining.

Customer value and inactivity risk turn out to be correlated at **-0.87** -
unsurprisingly, since both are built from how often and how much someone buys.
Banding risk on the raw probability therefore put **zero** high-value customers
in the "high risk" band, and pushed 97% of the high-risk population into the
*lowest* value tier. The resulting campaign list would have spent the entire
budget on the least valuable customers.

The fix: band risk **relative to peers of similar value**. A customer is high
risk if they sit in the riskiest 20% *of their own value tier*. A high-value
customer at p=0.55 is genuinely alarming even though 0.55 is unremarkable across
the whole base.

Both readings are kept on the output: `risk_band_absolute` is the raw model
output, `risk_band` is the peer-relative one used for prioritisation.

In [8]:
master, summary = RS.run()

Customer master built: 4,334 customers (3,471 scored + 863 already dormant)

Customers by value tier x risk band:
risk_band            Low  Medium  High
customer_value_tier                   
High                 378     226   152
Medium               679     408   272
Low                  678     407   271

Value at Risk (GBP) by value tier x risk band:
risk_band                Low   Medium     High
customer_value_tier                           
High                 30242.0  72402.0  65573.0
Medium               80403.0  82634.0  51328.0
Low                  56908.0  29683.0  12719.0

Retention priority groups:
    retention_priority  customers  pct_customers  value_at_risk  avg_inactivity_probability
          1 - Critical        152            3.5       65573.39                        0.38
              2 - High        498           11.5      123729.71                        0.43
3 - Low-Cost Retention        271            6.3       12718.68                        0.80
           3

## 7. The priority matrix

In [9]:
scored = master[master.risk_band != 'Already Dormant']
display(pd.crosstab(scored.customer_value_tier, scored.risk_band)
        .reindex(['High','Medium','Low']).reindex(columns=['Low','Medium','High']))
display(pd.read_csv(config.REPORTS_DIR / 'retention_priority_summary.csv'))

risk_band,Low,Medium,High
customer_value_tier,,,
High,378,226,152
Medium,679,408,272
Low,678,407,271


,retention_priority,customers,value_at_risk,avg_inactivity_probability,recent_revenue,pct_customers,action
0,1 - Critical,152,65573.39,0.38,355320.94,3.5,Personal contact from account manager plus a t...
1,2 - High,498,123729.71,0.43,884594.29,11.5,Targeted offer on their usual categories
2,3 - Low-Cost Retention,271,12718.68,0.80,32222.49,6.3,Automated email or discount code only
3,3 - Monitor,408,82633.55,0.48,341655.04,9.4,Add to the standard nurture campaign and re-sc...
4,4 - Maintain,378,30242.23,0.04,2740055.52,8.7,Loyalty rewards and early access - protect the...
5,5 - Low Priority,1764,166994.82,0.49,898022.73,40.7,No action needed - keep in normal marketing
6,6 - Reactivation,863,0.00,NaN,0.00,19.9,Outside the 180-day scoring window - low-cost ...


## 8. Customer Value at Risk (Phase 9)

```
Value at Risk = P(inactive) x (spend in the 180-day window / 2)
```

The division by two puts the customer's spending rate on the same 90-day footing
as the prediction horizon.

**This is an exposure estimate for ranking customers, not a revenue forecast.**
It is not money the business is guaranteed to lose - a flagged customer may well
come back, with or without intervention.

Because the test snapshot's future actually happened, the assumption can be
scored rather than just asserted:

In [10]:
_ = RS.validate_value_at_risk()


Value at Risk validation (on the test snapshot, whose future is observed):
  predicted expected revenue GBP    1,415,515
  actual future revenue      GBP    2,515,102   -> estimate captures 56% of it
  The straight-line assumption is CONSERVATIVE: the observed window was a seasonal peak
  and retained customers grow their spend. Treat Value at Risk as a ranking tool, not a forecast.
            customers  mean_value_at_risk  actual_inactive_rate  historic_value_of_those_who_lapsed
var_decile                                                                                         
1                 278              22.978                 0.385                              5110.0
2                 277              49.257                 0.466                              9238.0
3                 278              64.029                 0.453                             17147.0
4                 277              78.966                 0.394                             17412.0
5            

Two honest observations:

1. The straight-line estimate captures only **56%** of the revenue that actually
   materialised - it is **conservative**, because the observed window was the
   Christmas peak and retained customers grow their spend.
2. The top Value-at-Risk decile has a *lower* inactivity rate than the bottom
   one. That is expected: Value at Risk is dominated by value, not probability.
   It still does its job - customers in the top decile who lapsed took **GBP 110k**
   of historic value with them, versus GBP 5k in the bottom decile.

So Value at Risk is a sound **ranking** tool and a poor **forecasting** tool, and
it is presented that way.

## 9. The answer: who to contact first

In [11]:
cols = ['CustomerID','country','segment','customer_value_tier','inactivity_probability',
        'value_at_risk','lifetime_orders','lifetime_revenue','lifetime_recency_days']
critical = master[master.retention_priority == '1 - Critical'].nlargest(15, 'value_at_risk')
critical[cols].reset_index(drop=True)

,CustomerID,country,segment,customer_value_tier,inactivity_probability,value_at_risk,lifetime_orders,lifetime_revenue,lifetime_recency_days
0,16000,United Kingdom,Champions,High,0.4215,2611.97,3,12393.70,2
1,12678,France,Loyal Customers,High,0.2847,1670.78,12,17628.46,41
2,18139,United Kingdom,Champions,High,0.3790,1599.07,6,8438.34,17
3,12432,Norway,Loyal Customers,High,0.5427,1372.85,5,5059.32,42
4,12557,Spain,Champions,High,0.3819,1362.68,5,11990.96,15
5,17396,United Kingdom,Loyal Customers,High,0.3173,1097.11,7,7312.80,38
6,12409,Switzerland,At Risk,High,0.4316,1050.00,3,11072.67,78
7,16532,United Kingdom,Champions,High,0.3978,962.36,3,6748.80,17
8,12744,Singapore,Loyal Customers,High,0.3406,842.74,4,9120.39,55
9,18251,United Kingdom,Hibernating,High,0.3813,822.60,1,4314.72,86


## Conclusion

| Priority | Customers | Value at Risk | Action |
|---|---|---|---|
| **1 - Critical** | 152 | GBP 65.6k | Personal contact plus a tailored offer |
| **2 - High** | 498 | GBP 123.7k | Targeted offer / win-back with a deadline |
| **3 - Monitor** | 408 | GBP 82.6k | Standard nurture, re-score next month |
| **3 - Low-Cost Retention** | 271 | GBP 12.7k | Automated email only |
| **4 - Maintain** | 378 | GBP 30.2k | Loyalty rewards - protect the relationship |
| **5 - Low Priority** | 1,764 | GBP 167.0k | Normal marketing |
| **6 - Reactivation** | 863 | - | Already dormant beyond the scoring window |

**The headline:** 650 customers - 15% of the base - carry GBP 189k of the total
GBP 482k exposure. Starting with those 650, and handling the 152 Critical ones
personally, concentrates retention spending where the value actually is.